# POS Tagging - BERT Fine-Tuning

## Objective

This notebook trains BERT for Part-of-Speech (POS) tagging using the Universal Dependencies English EWT dataset.

Two approaches are evaluated:

- Full Fine-Tuning: all BERT parameters are trainable.
- Feature-Based: BERT parameters are frozen and only the classification head is trained.

The base model is `bert-base-uncased`.

A fixed random seed of 42 is used for reproducibility.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/BERT_FineTuning_Project"

import os
os.makedirs(f"{BASE_DIR}/models", exist_ok=True)
os.makedirs(f"{BASE_DIR}/checkpoints", exist_ok=True)

print(BASE_DIR)

/content/drive/MyDrive/BERT_FineTuning_Project


## Reproducibility and Configuration

Google Drive is used to store checkpoints and trained models.

A fixed random seed of 42 is used throughout the experiment.

Both approaches use the same pretrained model, `bert-base-uncased`, to provide a consistent comparison.

In [ ]:
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [ ]:
SEED = 42

import os
import random
import numpy as np
import torch
from transformers import set_seed

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

print("Seed configurada:", SEED)

Seed configurada: 42


## Dataset: UD English EWT

The Universal Dependencies English EWT dataset is used for Part-of-Speech tagging.

Each sentence is divided into tokens, and every token is assigned a Universal POS tag.

The dataset contains 17 POS categories, including nouns, verbs, adjectives, adverbs, pronouns, determiners, and punctuation.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "universal-dependencies/universal_dependencies",
    "en_ewt",
    revision="2.18"
)

print(dataset)

README.md:   0%|          | 0.00/87.2k [00:00<?, ?B/s]

parquet/en_ewt/dev.parquet: reconstructing file:   0%|          |  0.00B /  516kB            

parquet/en_ewt/dev.parquet: downloading bytes:           |  0.00B            

parquet/en_ewt/test.parquet: reconstructing file:   0%|          |  0.00B /  523kB            

parquet/en_ewt/test.parquet: downloading bytes:           |  0.00B            

...

parquet/en_ewt/train.parquet: downloading bytes:           |  0.00B            

Generating dev split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    dev: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2077
    })
    train: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 12544
    })
})


## Dataset Preparation

The dataset is divided into training, development, and test sets.

The original POS tag names are converted into numerical label IDs so they can be used by the BERT token classification model.

In [ ]:
print(dataset["train"][0]["tokens"])
print(dataset["train"][0]["upos"])

['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
['PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'PROPN', 'PROPN', 'PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'PROPN', 'PUNCT', 'ADP', 'DET', 'ADJ', 'NOUN', 'PUNCT']


In [ ]:
label_names = [
    "ADJ",
    "ADP",
    "ADV",
    "AUX",
    "CCONJ",
    "DET",
    "INTJ",
    "NOUN",
    "NUM",
    "PART",
    "PRON",
    "PROPN",
    "PUNCT",
    "SCONJ",
    "SYM",
    "VERB",
    "X"
]

label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for i, label in enumerate(label_names)}

print(label2id)

{'ADJ': 0, 'ADP': 1, 'ADV': 2, 'AUX': 3, 'CCONJ': 4, 'DET': 5, 'INTJ': 6, 'NOUN': 7, 'NUM': 8, 'PART': 9, 'PRON': 10, 'PROPN': 11, 'PUNCT': 12, 'SCONJ': 13, 'SYM': 14, 'VERB': 15, 'X': 16}


## POS Label Mapping

The model uses 17 Universal POS tags:

- ADJ
- ADP
- ADV
- AUX
- CCONJ
- DET
- INTJ
- NOUN
- NUM
- PART
- PRON
- PROPN
- PUNCT
- SCONJ
- SYM
- VERB
- X

Each token receives one of these labels during training.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )

    labels = []

    for i, word_labels in enumerate(examples["upos"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(
                    label2id[word_labels[word_idx]]
                )

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

## Tokenization and Label Alignment

The BERT tokenizer converts each sentence into subword tokens.

Because one original word can be divided into multiple subwords, the POS label is assigned to the first subword.

Additional subwords are ignored during loss calculation using the label `-100`.

This keeps the POS labels aligned with the original tokens.

In [ ]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Map:   0%|          | 0/12544 [00:00<?, ? examples/s]

DatasetDict({
    dev: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2077
    })
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 12544
    })
})


In [ ]:
from transformers import AutoModelForTokenClassification

model_full = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        for p, l in zip(prediction, label):

            if l != -100:
                true_predictions.append(p)
                true_labels.append(l)

    accuracy = accuracy_score(
        true_labels,
        true_predictions
    )

    return {
        "accuracy": accuracy
    }

## Full Fine-Tuning

In the Full Fine-Tuning approach, all BERT parameters are trainable.

The model can adapt its pretrained representations to the POS tagging task.

The learning rate used is `2e-5`.

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [ ]:
from transformers import TrainingArguments

training_args_full = TrainingArguments(
    output_dir=f"{BASE_DIR}/checkpoints/pos_full",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    seed=SEED,
    data_seed=SEED,

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True
)

In [ ]:
from transformers import Trainer

trainer_full = Trainer(
    model=model_full,
    args=training_args_full,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["dev"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [ ]:
trainer_full.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.107334,0.120495,0.968029
2,0.059637,0.111484,0.971727
3,0.042513,0.115995,0.972324


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=2352, training_loss=0.13255008031316354, metrics={'train_runtime': 431.9338, 'train_samples_per_second': 87.124, 'train_steps_per_second': 5.445, 'total_flos': 996720981264384.0, 'train_loss': 0.13255008031316354, 'epoch': 3.0})

In [ ]:
results_full = trainer_full.evaluate()

print("POS — Full Fine-tuning")
print("Accuracy:", results_full["eval_accuracy"])

Training Loss,Validation Loss,Epoch,Accuracy
0.042513,0.115995,3,0.972324


POS — Full Fine-tuning
Accuracy: 0.972323842850326


In [ ]:
MODEL_DIR_FULL = f"{BASE_DIR}/models/ud_ewt_bert_pos_full"

trainer_full.save_model(MODEL_DIR_FULL)
tokenizer.save_pretrained(MODEL_DIR_FULL)

print("POS Full guardado en:")
print(MODEL_DIR_FULL)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

POS Full guardado en:
/content/drive/MyDrive/BERT_FineTuning_Project/models/ud_ewt_bert_pos_full


In [ ]:
!ls -lah "$BASE_DIR/models/ud_ewt_bert_pos_full"

total 417M
drwx------ 2 root root 4.0K Sep 25 22:06 .
drwx------ 5 root root 4.0K Sep 25 22:06 ..
-rw------- 1 root root 1.4K Sep 25 22:06 config.json
-rw------- 1 root root 416M Sep 25 22:06 model.safetensors
-rw------- 1 root root  351 Sep 25 22:06 tokenizer_config.json
-rw------- 1 root root 695K Sep 25 22:06 tokenizer.json
-rw------- 1 root root 5.2K Sep 25 22:06 training_args.bin


## Feature-Based Training

In the Feature-Based approach, the pretrained BERT parameters are frozen.

Only the POS classification head is trained.

This allows BERT to act as a fixed feature extractor.

The learning rate used for the trainable classification head is `1e-3`.

In [ ]:
model_feature = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

In [ ]:
for param in model_feature.base_model.parameters():
    param.requires_grad = False

for param in model_feature.classifier.parameters():
    param.requires_grad = True

In [ ]:
def compute_metrics_feature(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        for p, l in zip(prediction, label):

            if l != -100:
                true_predictions.append(p)
                true_labels.append(l)

    accuracy = accuracy_score(
        true_labels,
        true_predictions
    )

    return {
        "accuracy": accuracy
    }

In [ ]:
training_args_feature = TrainingArguments(
    output_dir=f"{BASE_DIR}/checkpoints/pos_feature",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    seed=SEED,
    data_seed=SEED,

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True
)

In [ ]:
trainer_feature = Trainer(
    model=model_feature,
    args=training_args_feature,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["dev"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics_feature
)

In [ ]:
trainer_feature.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.334340,0.289297,0.920312
2,0.269164,0.249689,0.929617
3,0.260443,0.242253,0.930571


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=2352, training_loss=0.36035762392744725, metrics={'train_runtime': 174.1149, 'train_samples_per_second': 216.133, 'train_steps_per_second': 13.508, 'total_flos': 996720981264384.0, 'train_loss': 0.36035762392744725, 'epoch': 3.0})

In [ ]:
results_feature = trainer_feature.evaluate()

print("POS — Feature-based")
print("Accuracy:", results_feature["eval_accuracy"])

Training Loss,Validation Loss,Epoch,Accuracy
0.260443,0.242253,3,0.930571


POS — Feature-based
Accuracy: 0.9305710195641801


## Evaluation

The models are evaluated using token-level accuracy.

Accuracy measures the percentage of tokens for which the predicted POS tag matches the correct label.

In [ ]:
MODEL_DIR_FEATURE = f"{BASE_DIR}/models/ud_ewt_bert_pos_feature"

trainer_feature.save_model(MODEL_DIR_FEATURE)
tokenizer.save_pretrained(MODEL_DIR_FEATURE)

print("POS Feature guardado en:")
print(MODEL_DIR_FEATURE)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

POS Feature guardado en:
/content/drive/MyDrive/BERT_FineTuning_Project/models/ud_ewt_bert_pos_feature


In [ ]:
!ls -lah "$BASE_DIR/models/ud_ewt_bert_pos_feature"

total 417M
drwx------ 2 root root 4.0K Sep 25 22:12 .
drwx------ 6 root root 4.0K Sep 25 22:12 ..
-rw------- 1 root root 1.4K Sep 25 22:12 config.json
-rw------- 1 root root 416M Sep 25 22:12 model.safetensors
-rw------- 1 root root  351 Sep 25 22:12 tokenizer_config.json
-rw------- 1 root root 695K Sep 25 22:12 tokenizer.json
-rw------- 1 root root 5.2K Sep 25 22:12 training_args.bin


In [ ]:
MODEL_DIR_FEATURE = f"{BASE_DIR}/models/ud_ewt_bert_pos_feature"

trainer_feature.save_model(MODEL_DIR_FEATURE)
tokenizer.save_pretrained(MODEL_DIR_FEATURE)

print("POS Feature guardado en:")
print(MODEL_DIR_FEATURE)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

POS Feature guardado en:
/content/drive/MyDrive/BERT_FineTuning_Project/models/ud_ewt_bert_pos_feature


## Final Results

### Full Fine-Tuning

- Training Loss: 0.042513
- Validation Loss: 0.115995
- Accuracy: 97.23%

### Feature-Based

- Training Loss: 0.260443
- Validation Loss: 0.242253
- Accuracy: 93.06%

Full Fine-Tuning achieved higher accuracy because the BERT parameters were allowed to adapt to the POS tagging task.